# Part 1: Distributions — Reading the Shape of Data
**⏱ This section takes approximately 30 minutes.**

---

## Scenario: Tuesday — Explaining "Skewed" to Priya

Sarah arrives Tuesday morning ready to address Priya's question from Friday.
Her plan: pull all 10,000 polarity scores and *show* Priya what the data actually looks like,
rather than summarising it as one number.

But as soon as she plots the histogram, she realises she has a new problem:
the data is not a nice bell curve. It leans. Priya is going to ask what that means.

> *"The chart shows most reviews clustering on the left, with a longer tail on the right. What does that mean for the 60% figure?"*
> — Priya, after seeing the plot

**By the end of this notebook you will be able to:**
- Describe the shape of any distribution using the right vocabulary
- Explain why mean and median diverge in skewed data
- Calculate and interpret Z-scores to identify unusual values
- Recognise when distribution shape affects modelling decisions

In [ ]:
# Setup — run this cell first
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print("✅ Libraries loaded — you're ready to go!")


## Rebuilding Sarah's Dataset

We start from the same synthetic data as the pre-class hook notebook —
Sarah's 10,000 polarity scores from the L01 sentiment model run.


In [ ]:
# Recreate the NorthStar polarity scores (same seed as 01_monday_morning.ipynb)
np.random.seed(42)
n_reviews = 10_000

# 60% positive, right-skewed: mildly-positive bulk + long positive tail,
# smaller negative cluster nearer zero
positive = np.random.exponential(scale=0.35, size=6_000)
positive = np.clip(positive, 0.001, 1.0)
negative = -np.random.exponential(scale=0.20, size=4_000)
negative = np.clip(negative, -1.0, -0.001)
polarity_scores = np.concatenate([positive, negative])
np.random.shuffle(polarity_scores)

# Also generate synthetic review lengths (word count per review)
review_lengths = np.random.exponential(scale=45, size=n_reviews).astype(int) + 5
review_lengths = np.clip(review_lengths, 5, 500)

reviews_df = pd.DataFrame({
    'polarity': polarity_scores,
    'word_count': review_lengths,
    'label': np.where(polarity_scores > 0, 'POSITIVE', 'NEGATIVE')
})

print(f"Dataset ready: {len(reviews_df):,} reviews")
print(reviews_df.describe().round(3))

**<span style="color:green">[Opus 4.8]</span> Reading Sarah's dataset — what every number above means:**

The `.describe()` table is a *numerical summary* of the 10,000 reviews. Each column is a variable; each row is a statistic. Here is how to read it:

| Statistic | What it is | `polarity` | `word_count` |
|---|---|---|---|
| **count** | How many non-missing values | 10,000 | 10,000 |
| **mean** | The arithmetic average | 0.121 | 49.68 words |
| **std** | Standard deviation — typical spread around the mean | 0.356 | 45.11 words |
| **min** | Smallest value | −1.000 | 5 words |
| **25%** | First quartile (Q1) — 25% of values fall below this | −0.092 | 17 words |
| **50%** | Median (Q2) — the middle value | 0.062 | 36 words |
| **75%** | Third quartile (Q3) — 75% of values fall below this | 0.306 | 67 words |
| **max** | Largest value | 1.000 | 415 words |

**Polarity score (the model's sentiment, −1 = very negative, +1 = very positive):**
- The **mean (0.121) sits above the median (0.062)** — the classic fingerprint of a **right-skewed** distribution. A long tail of strongly-positive reviews drags the average up, even though the typical review is only mildly positive.
- The **25th percentile is negative (−0.092)** but the **median is positive (0.062)**, which tells you the positive/negative split lands somewhere between the 25% and 50% marks — consistent with the dataset being built as ~60% positive, ~40% negative.
- `min` and `max` hit exactly **−1.000 and 1.000** because the data was clipped to that range when generated — those are hard boundaries, not natural extremes.

**Word count (length of each review):**
- **Even more strongly right-skewed:** the mean (49.7) is well above the median (36), and the max (415) towers over the 75th percentile (67). Most reviews are short; a handful of very long ones stretch the tail.
- The **std (45) is almost as large as the mean (50)** — a high spread-to-centre ratio that itself signals a skewed, non-normal variable. For a bell curve you'd expect the std to be a much smaller fraction of the mean.
- **Half of all reviews are between 17 and 67 words** (Q1 to Q3 — the *interquartile range*, the middle 50%). That IQR is a more honest "typical" range here than mean ± std, which would dip below the 5-word minimum.

**The one takeaway:** for both variables the **mean > median**, so quoting the mean alone overstates the "typical" review. This is exactly the skew Priya spotted — and the reason the rest of this notebook leans on the *median* and the *shape* of the distribution rather than a single average.

*<span style="color:green">[Opus 4.8] — end of note</span>*


## 🎯 What is a Distribution?

**The idea in plain English:**
> A distribution is the pattern of how often each value appears in a dataset. It answers the question: "What does this data look like — where do values cluster, and how spread out are they?"

**The coffee shop analogy:** A coffee shop records its daily customer count for a year. Some days are quiet (50 customers), some are very busy (250), but most days fall somewhere in between. If you drew a bar chart of every day's count, the shape of that chart is the distribution. It tells you what a "typical" day looks like, and which days were unusual.

**Why it matters for ML:** Most ML algorithms behave differently depending on the shape of the data they receive. A linear model implicitly assumes roughly normal (bell-shaped) features. A gradient-boosted tree is more robust to shape, but still affected by extreme outliers. Knowing the distribution is step one of any data analysis.

**Mean vs median** — the **mean** is the arithmetic average (sum of all values ÷ count); the **median** is the middle value when the data is sorted. The gap between them tells you about the skew: where the mean is *above* the median, the data is right-skewed; where the mean is *below* the median, the data is left-skewed.

---

### The three shapes you will see most often

| Shape | What it looks like | Real-world examples |
|---|---|---|
| **Normal (bell curve)** | Symmetric, peak in the middle, thin tails | Heights, test scores, measurement errors |
| **Right-skewed** | Peak on the left, long tail to the right | Incomes, house prices, review lengths |
| **Left-skewed** | Peak on the right, long tail to the left | Exam scores on an easy exam, age at retirement |


## ⏸️ Pause and Predict

We're about to plot two distributions side by side:
1. The polarity scores of NorthStar reviews
2. The word counts of those same reviews

**Before running the cell below, predict:**
- Which of the three shapes (normal / right-skewed / left-skewed) do you expect each distribution to have?
- For each distribution: will the mean be higher, lower, or roughly equal to the median?

*Write your prediction here (double-click this cell to edit):*


In [ ]:
# Plot the polarity and review-length distributions side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Left panel: polarity scores ---
ax1 = axes[0]
ax1.hist(reviews_df['polarity'], bins=60, color='steelblue', edgecolor='white', alpha=0.85)
ax1.axvline(reviews_df['polarity'].mean(), color='orange', linewidth=2,
            label=f"Mean: {reviews_df['polarity'].mean():.3f}")
ax1.axvline(reviews_df['polarity'].median(), color='green', linewidth=2, linestyle='--',
            label=f"Median: {reviews_df['polarity'].median():.3f}")
ax1.set_xlabel("Polarity Score (−1 to +1)")
ax1.set_ylabel("Number of Reviews")
ax1.set_title("Review Polarity Scores")
ax1.legend()

# --- Right panel: review word counts ---
ax2 = axes[1]
ax2.hist(reviews_df['word_count'], bins=60, color='coral', edgecolor='white', alpha=0.85)
ax2.axvline(reviews_df['word_count'].mean(), color='orange', linewidth=2,
            label=f"Mean: {reviews_df['word_count'].mean():.0f} words")
ax2.axvline(reviews_df['word_count'].median(), color='green', linewidth=2, linestyle='--',
            label=f"Median: {reviews_df['word_count'].median():.0f} words")
ax2.set_xlabel("Review Length (words)")
ax2.set_ylabel("Number of Reviews")
ax2.set_title("Review Word Count Distribution")
ax2.legend()

plt.tight_layout()
plt.show()

print("Polarity scores:")
print(f"  Mean: {reviews_df['polarity'].mean():.3f}  |  Median: {reviews_df['polarity'].median():.3f}")
print(f"  Skew: {reviews_df['polarity'].skew():.3f}  (positive = right-skewed; negative = left-skewed)")
print()
print("Review lengths:")
print(f"  Mean: {reviews_df['word_count'].mean():.1f} words  |  Median: {reviews_df['word_count'].median():.0f} words")
print(f"  Skew: {reviews_df['word_count'].skew():.3f}")


### 💡 What do you notice?

- **Polarity scores are mildly right-skewed** — there is a long tail to the right (very positive reviews) and most reviews cluster in the mildly-positive range. The mean is pulled upward by those extreme positive values.
- **Review lengths are strongly right-skewed** — most reviews are short (under 60 words), but a long tail of very long reviews stretches to the right. The mean length is noticeably higher than the median because those long reviews pull the average up.

**Back to our scenario:**
> If Priya asks "how long is a typical review?", the median (around 40 words) is more honest than the mean (pulled higher by a few very long complaints). This is exactly the kind of nuance that separates a useful data analyst from one who just runs code.

## ✅ Section Summary

| Concept | What it means | Real-world use |
|---|---|---|
| **Distribution** | The pattern of how often each value appears in a dataset | Understanding what "normal" looks like before flagging anomalies |
| **Normal (bell curve)** | Symmetric, mean ≈ median, thin tails | Heights, test scores, measurement errors |
| **Right-skewed** | Long tail to the right, mean > median | Incomes, review lengths, house prices |
| **Left-skewed** | Long tail to the left, mean < median | Easy exam scores, age at retirement |
| **Mean vs Median** | Mean is sensitive to extremes; median is resistant | Always check both when the distribution might be skewed |

**Key insight for our scenario:**
> Sarah's polarity scores are mildly right-skewed — the median is the more representative summary for Priya. *Knowing the shape of the data is step one of any serious ML project.*

---
**Up next → Part 2:** Wednesday — Sarah labels 200 reviews by hand and learns to put a confidence bracket around her 84% accuracy.
Open `03_confidence_intervals.ipynb`

---

## 🟡 Extension — self-study after class

*Skipping this section will not affect your understanding of later lessons. Come back to it when you have time and want to go deeper.*

This optional section covers:
- **Z-scores** — how to flag unusual values (a key feature-standardisation technique used in ML)

## 🎯 Z-scores: Measuring How Unusual a Value Is

**The idea in plain English:**
> A Z-score answers: *"How many standard deviations away from the average is this value?"*
> Z = 0 means exactly average. Z = 2 means "two standard deviations above average." Z = −3 means "very far below average."

**Standard deviation — a quick recap:**
Standard deviation measures how spread out values are around the mean. A small standard deviation means most values sit close to the average; a large one means they are spread widely. For example, two classes both average 70 on a test: one has scores ranging 68–72 (tiny standard deviation), the other ranges 40–100 (large standard deviation). The Z-score uses this spread as its unit of measurement, so you can instantly see how unusual any single value is relative to the rest of the data.

**Formula:**
```
z = (value − mean) / standard deviation
```

**The exam score analogy:** You score 85 on an exam. Is that good?
- If the class average was 60 and the standard deviation was 10 → Z = (85−60)/10 = **2.5** — top 1% of the class.
- If the class average was 82 and the standard deviation was 2 → Z = (85−82)/2 = **1.5** — above average, but not exceptional.
The raw score means nothing without context; the Z-score provides that context instantly.

**Why it matters for ML:**
Z-scores are the foundation of *feature standardisation* — one of the most common preprocessing steps before training a model. This works for **any** distribution shape.

Z-scores are also used in outlier detection, where |Z| > 3 is a common rule of thumb for flagging unusual values. **Important caveat:** this threshold only works reliably when the data is approximately normally distributed. For skewed data, the rule misleads — the skew shifts where extreme values land relative to the mean and standard deviation, so the |Z| > 3 boundary is no longer a meaningful cut-off. We will see this play out in the example below.

**<span style="color:green">[Opus 4.8]</span> Deeper detail on the Z-score:**

*(The intro above recaps what standard deviation is and warns that |Z| > 3 misleads on skewed data. These notes go one level deeper on the mechanics.)*

- **Read the formula term by term.** The **numerator** `value − mean` is how far the point sits from the centre, in the original units (positive = above average, negative = below). The **denominator** `standard deviation` is the dataset's "natural ruler" — the typical distance values sit from the mean.
- **Dividing by the standard deviation** converts that gap into a *count of rulers*, which is why a Z-score has **no units** and can be compared across completely different variables (height vs income vs polarity).
- **Quick reading guide:** Z ≈ 0 is dead average; ±1 is normal variation; ±2 is starting to look unusual; **±3 or beyond is an outlier worth investigating** (on roughly normal data — see the intro's caveat).
- **Where the ±1 / ±2 / ±3 cut-offs come from — the 68–95–99.7 rule:** on a bell curve about **68%** of values fall within ±1 SD, **95%** within ±2, and **99.7%** within ±3. So |Z| > 3 happens under ~0.3% of the time *by chance* — that is the origin of the "outlier" threshold. (The next cells unpack where those percentages come from.)
- **In practice (ML):** standardising every feature to Z-scores — mean 0, std 1 — is exactly what scikit-learn's `StandardScaler` does. It stops a feature measured in thousands from dominating one measured in decimals when training linear models, k-NN, SVMs, or neural networks.

*<span style="color:green">[Opus 4.8] — end of note</span>*


**<span style="color:green">[Opus 4.8]</span> Standard deviation — how the number is actually computed, and a footnote on "top 1%":**

The intro recaps *what* standard deviation means (spread around the mean). Here is *how* it is calculated, plus a precise look at the exam-analogy claim.

**How is its value derived?** Five mechanical steps — worked on the example `[2, 4, 4, 4, 5, 5, 7, 9]`:

| Step | What you do | This example |
|---|---|---|
| 1. Mean | Average all the values | (2+4+4+4+5+5+7+9) / 8 = **5** |
| 2. Deviations | Subtract the mean from each value | −3, −1, −1, −1, 0, 0, +2, +4 |
| 3. Square them | Square each deviation (so −/+ don't cancel) | 9, 1, 1, 1, 0, 0, 4, 16 |
| 4. Variance | Average the squared deviations | 32 / 8 = **4** |
| 5. Std deviation | Square-root the variance (back to original units) | √4 = **2** |

> **÷N vs ÷(N−1):** a full *population* divides by N (step 4); a *sample* divides by **N−1** (Bessel's correction). pandas `.std()` uses N−1, numpy `.std()` uses N — which is why the same data can show a slightly different std depending on the tool. On 10,000 rows the difference is negligible.

**"1 standard deviation" = one step of typical spread.** With mean 5 and std 2 above, +1 SD = 7 and −1 SD = 3; on a bell curve that ±1 band holds ~68% of values. That is the ruler a Z-score counts in.

**Footnote on the exam analogy:** the intro calls Z = 2.5 "top 1% of the class." Strictly it is the **top ~0.6%** — in a normal distribution only about 0.6% of values sit beyond Z = 2.5 (≈2.3% beyond 2.0, ≈0.13% beyond 3.0). "Top 1%" is a friendly rounding; the exact figure comes from the tail areas explained in the next cell.

*<span style="color:green">[Opus 4.8] — end of note</span>*


**<span style="color:green">[Opus 4.8]</span> Where do the tail percentages (~2.3% / ~0.6% / ~0.13%) actually come from?**

**The big idea: percentages are areas under the curve.**
A distribution can be drawn as a curve. The **total area under it equals 1 (= 100% of the data)**. So *any* "what fraction of values..." question becomes a *"how much area..."* question. "What % score higher than Z = 2.5?" is literally **the area under the bell curve to the right of 2.5.**

**The standard normal curve.**
Once you convert to Z-scores, every normal distribution becomes the *same* fixed curve — the **standard normal**, with mean 0 and std 1. Its exact equation is:

```
φ(z) = (1 / √(2π)) · e^(−z² / 2)
```

You never need to use that formula by hand. What matters is the area under it.

**Φ(z) — the cumulative area from the left.**
Mathematicians define **Φ(z)** ("big phi") as *the area to the LEFT of z* — i.e. the fraction of values **at or below** z. The area in the **right tail** (values *above* z, the "how many beat me") is then just the leftover:

```
P(Z > z)  =  1 − Φ(z)
```

There is no neat algebra for Φ(z) — it can only be computed numerically — which is why textbooks print **"Z-tables"** and code uses a built-in function. The classic values:

| z | Φ(z) = area to the left | right tail = 1 − Φ(z) | rounded |
|---|---|---|---|
| 1.0 | 0.8413 | 0.1587 | ~16% |
| 2.0 | 0.9772 | 0.0228 | **~2.3%** |
| 2.5 | 0.9938 | 0.0062 | **~0.6%** |
| 3.0 | 0.9987 | 0.0013 | **~0.13%** |

That right-tail column is exactly the table in the cell above. So **~2.3% of people score beyond Z = 2.0, ~0.6% beyond 2.5, ~0.13% beyond 3.0.**

**Sanity check against the 68–95–99.7 rule.**
The rule says **95.45%** of values sit *within* ±2 std. The remaining `100 − 95.45 = 4.55%` lives in the *two* tails combined. By symmetry each tail holds half: `4.55% / 2 = 2.275% ≈ 2.3%` — matching the table. Likewise ±3 leaves `100 − 99.73 = 0.27%`, split into `0.135% ≈ 0.13%` per tail. (Z = 2.5 isn't one of the "round" rule values, so for it you must look up Φ(2.5) rather than derive it from the rule.)

**How you'd compute it yourself** (no table needed):

```python
from scipy.stats import norm
norm.sf(2.5)        # 0.0062  — "survival function" = right-tail area = 1 − Φ(z)
1 - norm.cdf(2.5)   # 0.0062  — same thing, written out

# Pure standard library, no scipy:
import math
def right_tail(z):
    return 0.5 * (1 - math.erf(z / math.sqrt(2)))   # erf is the built-in error function
right_tail(2.5)     # 0.0062
```

**The one-line summary:** a percentage of the population *is* an area under the distribution curve; for the standard normal that area is the function Φ(z), and `1 − Φ(z)` gives the fraction scoring above any Z — which is why Z = 2.5 lands at "top ~0.6%."

*<span style="color:green">[Opus 4.8] — end of note</span>*


In [ ]:
# [Opus 4.8] Visual proof: a tail percentage IS the shaded area under the standard normal curve
import math

# --- The two building blocks (pure standard library, no scipy needed) ---
def pdf(z):                       # height of the standard normal curve at z
    return (1 / math.sqrt(2 * math.pi)) * math.exp(-z**2 / 2)

def right_tail(z):                # area to the RIGHT of z  =  P(Z > z)  =  1 - Phi(z)
    return 0.5 * (1 - math.erf(z / math.sqrt(2)))

# --- Print the tail percentages we used above ---
print("z     P(Z > z)   = top ...% scored higher")
for z in [1.0, 2.0, 2.5, 3.0]:
    print(f"{z:>3}   {right_tail(z):.4f}     ~{right_tail(z)*100:.2f}%")
print()

z_mark = 2.5                       # the exam example: 85 marks -> Z = 2.5
print(f"Z = {z_mark}: only {right_tail(z_mark)*100:.2f}% score higher "
      f"-> you beat {(1-right_tail(z_mark))*100:.2f}% of the class")

# --- Draw the curve and shade the right tail beyond z_mark ---
xs = np.linspace(-4, 4, 400)
ys = [pdf(x) for x in xs]

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(xs, ys, color='navy', linewidth=2, label='standard normal curve')

# shade everything to the right of z_mark = the tail probability
tail_x = np.linspace(z_mark, 4, 120)
ax.fill_between(tail_x, [pdf(x) for x in tail_x], color='crimson', alpha=0.6,
                label=f'area right of Z={z_mark}  =  {right_tail(z_mark)*100:.2f}%')

# light bands for the 68-95-99.7 rule
for k, c in [(1, '0.85'), (2, '0.92')]:
    band = np.linspace(-k, k, 100)
    ax.fill_between(band, [pdf(x) for x in band], color=c, zorder=0)

ax.axvline(0, color='grey', linewidth=1, linestyle=':')
ax.axvline(z_mark, color='crimson', linewidth=1.5, linestyle='--')
ax.annotate('your score\n(Z = 2.5)', xy=(z_mark, pdf(z_mark)),
            xytext=(2.9, 0.18), color='crimson',
            arrowprops=dict(arrowstyle='->', color='crimson'))
ax.set_xlabel("Z-score (standard deviations from the mean)")
ax.set_ylabel("relative likelihood (curve height)")
ax.set_title("Total area under the curve = 1 (100%).  The red sliver is the top ~0.6%.")
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()
print("# [Opus 4.8] — end of note")


In [ ]:
# Compute Z-scores for the polarity scores
pol_mean = polarity_scores.mean()
pol_std  = polarity_scores.std()

reviews_df['z_score'] = (reviews_df['polarity'] - pol_mean) / pol_std

# Show some examples
print(f"Population mean: {pol_mean:.4f}")
print(f"Population std:  {pol_std:.4f}")
print()
print("Example Z-scores:")
examples = reviews_df.nlargest(3, 'z_score')[['polarity','z_score','label']].round(3)
examples2 = reviews_df.nsmallest(3, 'z_score')[['polarity','z_score','label']].round(3)
print("  Most positive reviews:")
print(examples.to_string(index=False))
print()
print("  Most negative reviews:")
print(examples2.to_string(index=False))


In [ ]:
# Identify outliers using Z-score threshold
outlier_threshold = 3.0
outliers = reviews_df[np.abs(reviews_df['z_score']) > outlier_threshold]
n_pos_outliers = (reviews_df['z_score'] >  outlier_threshold).sum()
n_neg_outliers = (reviews_df['z_score'] < -outlier_threshold).sum()

print(f"Reviews with |Z| > {outlier_threshold}: {len(outliers)} ({len(outliers)/len(reviews_df):.2%})")
print(f"  ...of which {n_pos_outliers} are unusually POSITIVE (Z > +3)")
print(f"  ...and       {n_neg_outliers} are unusually NEGATIVE (Z < -3)")
print()

# Visualise Z-score distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
ax1.hist(reviews_df['z_score'], bins=60, color='steelblue', edgecolor='white', alpha=0.8)
ax1.axvline(-3, color='red', linewidth=2, linestyle='--', label='|Z| = 3 threshold')
ax1.axvline(3, color='red', linewidth=2, linestyle='--')
ax1.set_xlabel("Z-score")
ax1.set_ylabel("Count")
ax1.set_title("Z-score Distribution\n(Polarity Scores Standardised)")
ax1.legend()

ax2 = axes[1]
ax2.scatter(reviews_df.index[:500], reviews_df['z_score'][:500],
            c=np.abs(reviews_df['z_score'][:500]) > 3,
            cmap='coolwarm', alpha=0.4, s=10)
ax2.axhline(3, color='red', linewidth=1.5, linestyle='--', label='|Z| = 3')
ax2.axhline(-3, color='red', linewidth=1.5, linestyle='--')
ax2.set_xlabel("Review index (first 500)")
ax2.set_ylabel("Z-score")
ax2.set_title("Z-scores for First 500 Reviews\n(Red = outlier threshold)")
ax2.legend()

plt.tight_layout()
plt.show()

print("Z-score range in this dataset:")
print(f"  Most positive Z: {reviews_df['z_score'].max():.3f}  (hard ceiling — polarity model caps at +1.0)")
print(f"  Most negative Z: {reviews_df['z_score'].min():.3f}")
print()
print("Note: this right-skewed distribution with a ceiling at +1.0 is not well-suited")
print("for Z-score outlier detection. See the observations cell below for why.")

### 💡 What do you notice?

- **Z-scores standardise the data** — regardless of the original scale, the distribution is recentred at 0 and measured in standard deviation units. This is useful for feature standardisation in ML pipelines and works for any distribution shape.
- **The outliers are all on the negative side** — there's no Z above +3, but a handful of reviews sit below −3. The reason is structural: the sentiment model caps scores at +1.0, so the highest achievable Z-score for any positive review is approximately +2.5. No amount of enthusiasm can push a review past that ceiling.
- **⚠️ This example reveals why |Z| > 3 is a poor outlier rule for skewed data.** Two properties of this dataset break the assumption: (1) the distribution is right-skewed, not normal, and (2) there is a hard ceiling at +1.0 that prevents extreme positive Z-scores from ever forming. The asymmetric result — negatives flagged, positives never flagged — is an artefact of these properties, not a genuine signal. For skewed data, **IQR-based outlier detection** (using the interquartile range) is more reliable because it makes no assumption about distribution shape.

**Key takeaway:**
> Check the distribution shape *before* choosing an outlier detection method. Z-score thresholds are designed for approximately normal data. Sarah's polarity scores call for a different approach.